# ReLive Knowledge Base Test Harness

This notebook tests the full write/read workflow:

1. Load sample texts from `data/`
2. Write extracted knowledge into Neo4j using the writer graph
3. Run chat-like read queries against the knowledge base

Optional: choose a provider with `PROVIDER = "openai"` or `PROVIDER = "gemini"`, or use `PROVIDER = "none"` for deterministic local mode.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

from src.logic.orchestrator import AgentOrchestrator
from src.ai.providers import (
    GeminiEmbedderClient,
    GeminiLLMClient,
    GeminiProviderConfig,
    OpenAIEmbedderClient,
    OpenAILLMClient,
    OpenAIProviderConfig,
)

/home/r2/.pyenv/versions/relive_env/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
# Choose provider: "none", "openai", or "gemini".
PROVIDER = "gemini"

# Toggle to True to bypass external Neo4j and use in-memory debug backend.
USE_IN_MEMORY_DEBUG = True

import importlib
import os

if USE_IN_MEMORY_DEBUG:
    os.environ["NEO4J_DEBUG_IN_MEMORY"] = "true"

# Reload modules in case notebook kernel cached older code.
import src.database.config as db_config
import src.database.infrastructure.driver as neo_driver
import src.database.manager as db_manager
import src.logic.orchestrator as orchestrator_mod
importlib.reload(db_config)
importlib.reload(neo_driver)
importlib.reload(db_manager)
importlib.reload(orchestrator_mod)
neo_driver.Neo4jDriver._instance = None

llm = None
embedder = None
if PROVIDER == "openai":
    cfg = OpenAIProviderConfig.from_env()
    llm = OpenAILLMClient(cfg)
    embedder = OpenAIEmbedderClient(cfg)
elif PROVIDER == "gemini":
    cfg = GeminiProviderConfig.from_env()
    llm = GeminiLLMClient(cfg)
    embedder = GeminiEmbedderClient(cfg)

orchestrator = AgentOrchestrator(llm=llm, embedder=embedder)
await orchestrator.initialize()
print(f"Orchestrator initialized (provider={PROVIDER})")

/home/r2/Documents/Projects/ReLive/src/ai/providers.py:97: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai  # type: ignore[reportMissingImports]
DatabaseManager running in in-memory debug mode


Orchestrator initialized (provider=gemini)


In [3]:
data_dir = Path("data")
files = sorted(data_dir.glob("*.txt"))
print("Sample files:")
for f in files:
    print("-", f)

texts = [{"name": f.name, "text": f.read_text(encoding="utf-8")} for f in files]
print(f"Loaded {len(texts)} text documents")

Sample files:
- data/electronics_motor_overheat.txt
- data/python_etl_memory_spike.txt
- data/survival_cold_weather_fire.txt
Loaded 3 text documents


In [4]:
ingest_results = []
for item in texts:
    result = await orchestrator.run(mode="write", text=item["text"], environment_hint="auto")
    ingest_results.append({
        "file": item["name"],
        "knowledge_entry_id": result.get("persisted", {}).get("knowledge_entry_id"),
        "errors": result.get("errors", []),
    })

print(json.dumps(ingest_results, indent=2))

Reflection parse failed, using fallback grade: 1 validation error for WriterReflection
risk_factors
  Input should be a valid string [type=string_type, input_value=["Increased dust buildup ... smoothing parameters.'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type


[
  {
    "file": "electronics_motor_overheat.txt",
    "knowledge_entry_id": "6e82829c-e2f9-49fd-82ea-e6784f6fd226",
    "errors": []
  },
  {
    "file": "python_etl_memory_spike.txt",
    "knowledge_entry_id": "f56315e8-25f0-4960-a0bc-76da25f8c006",
    "errors": []
  },
  {
    "file": "survival_cold_weather_fire.txt",
    "knowledge_entry_id": "e57d8a0e-cf78-42fb-9291-080d2b1a16ba",
    "errors": []
  }
]


In [5]:
async def chat_turn(user_message: str, top_k: int | None = None) -> dict:
    """Run a single chat-like retrieval turn against the knowledge base."""
    response = await orchestrator.run(mode="read", text=user_message, top_k=top_k)
    return response.get("response", response)


def pretty_print_turn(query: str, payload: dict) -> None:
    print(f"USER: {query}\n")
    ranked = payload.get("ranked_results", [])
    if not ranked:
        print("ASSISTANT: No close matches found.")
        return

    best = ranked[0]
    print("ASSISTANT: Top match")
    print(json.dumps(best, indent=2))
    print("\nALTERNATIVES:")
    print(json.dumps(payload.get("alternatives", []), indent=2))

In [6]:
# Example chat-style conversation turns
queries = [
    "How can I reduce overheating in a dusty drone motor setup?",
]

for q in queries:
    payload = await chat_turn(q)  # top_k omitted -> adaptive traversal mode
    pretty_print_turn(q, payload)
    print("\n" + "=" * 80 + "\n")

USER: How can I reduce overheating in a dusty drone motor setup?

ASSISTANT: Top match
{
  "node_id": "ae596282-7900-4041-ab6e-e0bb915fe2eb",
  "text": "add vent filters, tune PWM duty-cycle ramping, and introduce firmware temperature smoothing",
  "avg_similarity": 0.8060382404914752,
  "concept_coverage": 1,
  "score": 0.6242267683440326,
  "concepts_matched": [
    "solution"
  ]
}

ALTERNATIVES:
[]




In [7]:
# Optional: manual interactive loop in notebook output
# Run this cell repeatedly with new prompt values.
user_query = "suggest alternatives for drone overheating in hot environment"
payload = await chat_turn(user_query)
pretty_print_turn(user_query, payload)

USER: suggest alternatives for drone overheating in hot environment

ASSISTANT: Top match
{
  "node_id": "4004870f-d442-421f-a9fa-960d129f7c77",
  "text": "intermittent motor shutdown after about 42 flight hours due to overheating causing protective throttling and occasional emergency stop",
  "avg_similarity": 0.8581830869917546,
  "concept_coverage": 1,
  "score": 0.6607281608942281,
  "concepts_matched": [
    "problem"
  ]
}

ALTERNATIVES:
[]


In [8]:
# Cleanup when done (recommended)
await orchestrator.close()
print("Orchestrator closed")

Orchestrator closed
